# Решение для соревнования `2026-nlp`

Задача: multilabel classification на 5 классов.

Финальная идея решения:

1. мягкая очистка новостных текстов;
2. transformer-модель `DeepPavlov/rubert-base-cased`;
3. 5-fold cross-validation;
4. подбор порогов по OOF-предсказаниям отдельно для каждого класса;
5. мягкая калибровка долей `target_2` и `target_3`;
6. сохранение итогового файла `sample_submission.csv`.

## 1. Установка и импорты


In [1]:
import torch
import transformers
import sklearn
import pandas as pd
import numpy as np

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)

torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
transformers: 5.0.0


In [3]:
import os
import re
import gc
import ast
import html
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import hamming_loss, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from scipy.sparse import hstack

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 250)

In [4]:
SEED = 322

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device: cuda
Tesla T4


In [5]:
def try_find_file(filename, roots):
    for root_dir in roots:
        if not os.path.exists(root_dir):
            continue
        for root, _, files in os.walk(root_dir):
            if filename in files:
                return os.path.join(root, filename)
    return None

roots = ["/kaggle/input", "."]

train_path = try_find_file("train.csv", roots)
test_path = try_find_file("test.csv", roots)
sample_path = try_find_file("sample_submission.csv", roots)

if train_path is None or test_path is None or sample_path is None:
    import kagglehub
    path = kagglehub.competition_download("2026-nlp")
    print("Path to competition files:", path)

    roots = [path, "/kaggle/input", "."]
    train_path = try_find_file("train.csv", roots)
    test_path = try_find_file("test.csv", roots)
    sample_path = try_find_file("sample_submission.csv", roots)

if train_path is None or test_path is None or sample_path is None:
    raise FileNotFoundError("Не удалось найти train.csv, test.csv или sample_submission.csv")

print("train:", train_path)
print("test:", test_path)
print("sample_submission:", sample_path)

train: /kaggle/input/competitions/2026-nlp/train.csv
test: /kaggle/input/competitions/2026-nlp/test.csv
sample_submission: /kaggle/input/competitions/2026-nlp/sample_submission.csv


In [6]:
train = pd.read_csv(train_path, sep="\t")
test = pd.read_csv(test_path, sep="\t")
sample_submission = pd.read_csv(sample_path)

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

train: (16701, 6)
test: (4969, 5)
sample_submission: (4969, 2)


,id,source,title,text,publication_date,target
0,0,Novosti,Рейтинг регионов по уровню закредитованности населения — 2019,"Средний <content>уровень</content> <source>ria.ru</source> 💰 закредитованности ⚡ <<em>content> <strong>россиян ✨ вырос</strong</em>> <![CDATA[ за 📍 <hr/> <p>2019 🏭 ⁉️ год с 44,9</p> &<b>copy;до 47,1</b>%. <article> 🗞️ <article> <div> 💰 <li> </con...",2019-12-23 00:00,"[0, 1, 0, 0, 0]"
1,1,Novosti,Названы самые закредитованные российские регионы,"МОСКВА, 23 дек — РИА Новости. Наиболее закредитованным субъектом Российской Федерации является Калмыкия , по объемам долга в среднем на человека лидируют северные регионы, меньше всего займов взяли на юге, свидетельствуют результаты исследо...",2019-12-23 00:21,"[0, 1, 0, 0, 0]"
2,2,Novosti,В России пройдут учения по обеспечению устойчивой работы рунета,"МОСКВА, 23 дек - РИА Новости. Всероссийские учения по обеспечению устойчивой <span class=""quote"">работы рунета и сети связи</span> общего &nbsp;пользования пройдут в понедельник, в них примут &laquo;участие операторы &ldquo;связи и органы ⭐ &ldq...",2019-12-23 00:28,"[1, 0, 0, 1, 0]"
3,3,Novosti,"Самолеты НАТО стали чаще летать у границ России, заявили в Балтфлоте","МОСКВА, 23 дек - РИА 📄 Новости. <hr> 📝 Интенсивность полетов у российских границ самолетов-разведчиков НАТО возросла &#xA0;в 2019 году более чем на треть, а </b> боевых самолетов <!-- source: ria --> - вдвое, &lsquo;🕒 сообщил командующий Балт...",2019-12-23 00:39,"[1, 0, 0, 0, 0]"
4,4,Novosti,"Сюткин оценил шутку Шнурова над обидевшим Гагарину участником ""Голоса""","МОСКВА, &nbsp;23 дек — РИА </content> Новости. 🌎 Певец Валерий <strong>Сюткин в интервью порталу Nation</strong> ⭐ News прокомментировал видео с шуткой <![CDATA[наставника шоу "" &ndash;Голос]]> "" 📝 &#8212;<span class=""quote"">Сергея Шнуров...",2019-12-23 00:50,"[0, 0, 0, 0, 0]"


,id,source,title,text,publication_date
0,16701,Spletnesti,Дым от австралийских лесных пожаров достиг Новой Зеландии. Огонь не утихает с,Власти направили военные корабли и авиацию для борьбы с огнём. ...,2020-01-01 07:35
1,16702,Spletnesti,Во Владивостоке в новогоднюю ночь сожгли фигуру мыши за 677 тысяч,"Светодиодную конструкцию не хотели убирать из-за суда с водителем, который её повредил. ...",2020-01-01 08:22
2,16703,Spletnesti,"Папа римский шлёпнул по руке женщину, которая схватила его на праздновании Нового года. А потом публично","Признав, что подал «плохой пример». ...",2020-01-01 15:37
3,16704,Spletnesti,Около 200 жителей закрытого Новоуральска встречали Новый Год на горнолыжном спуске,"С каждым годом количество горожан, выбирающих альтернативу традиционному новогоднему застолью, только увеличивается ...",2020-01-01 15:56
4,16705,Zholtosti,Как провести новогодние каникулы с ребенком по науке? Есть несколько идей,"🎊 Посмотреть на фейерверки с точки зрения науки . Это один из самых зрелищных способов провести химический анализ . Внутри снарядов, помимо прочих компонентов, упакованы металлы и соли металлов — именно они определяют, какого цвета будут искр...",2020-01-02 08:09


,id,target
0,16701,"[0,0,0,0,0]"
1,16702,"[0,0,0,0,0]"
2,16703,"[0,0,0,0,0]"
3,16704,"[0,0,0,0,0]"
4,16705,"[0,0,0,0,0]"


## 3. Target и базовый анализ

In [7]:
train["target_list"] = train["target"].apply(ast.literal_eval)
y = np.array(train["target_list"].tolist()).astype(np.float32)

for i in range(5):
    train[f"target_{i}"] = y[:, i]

train["labels_count"] = y.sum(axis=1).astype(int)

print("y shape:", y.shape)
print("target shares:", y.mean(axis=0))

display(train[["target", "target_0", "target_1", "target_2", "target_3", "target_4"]].head())

y shape: (16701, 5)
target shares: [0.4278187  0.13627927 0.10993354 0.07442668 0.0325729 ]


,target,target_0,target_1,target_2,target_3,target_4
0,"[0, 1, 0, 0, 0]",0.0,1.0,0.0,0.0,0.0
1,"[0, 1, 0, 0, 0]",0.0,1.0,0.0,0.0,0.0
2,"[1, 0, 0, 1, 0]",1.0,0.0,0.0,1.0,0.0
3,"[1, 0, 0, 0, 0]",1.0,0.0,0.0,0.0,0.0
4,"[0, 0, 0, 0, 0]",0.0,0.0,0.0,0.0,0.0


In [8]:
label_stats = pd.DataFrame({
    "label": [f"target_{i}" for i in range(5)],
    "positive_count": y.sum(axis=0).astype(int),
    "positive_share": y.mean(axis=0)
})

display(label_stats)

display(
    train["labels_count"]
    .value_counts()
    .sort_index()
    .rename_axis("labels_count")
    .reset_index(name="rows")
)

,label,positive_count,positive_share
0,target_0,7145,0.427819
1,target_1,2276,0.136279
2,target_2,1836,0.109934
3,target_3,1243,0.074427
4,target_4,544,0.032573


,labels_count,rows
0,0,5643
1,1,9209
2,2,1714
3,3,133
4,4,2


In [9]:
print("train source:")
display(train["source"].value_counts().reset_index())

print("test source:")
display(test["source"].value_counts().reset_index())

source_target = train.groupby("source")[[f"target_{i}" for i in range(5)]].mean()
display(source_target)

train source:


,source,count
0,Novosti,12759
1,Svezhesti,3942


test source:


,source,count
0,Novosti,1996
1,Zholtosti,1538
2,Spletnesti,867
3,Svezhesti,568


,target_0,target_1,target_2,target_3,target_4
source,,,,,
Novosti,0.431382,0.136296,0.109413,0.080884,0.013794
Svezhesti,0.416286,0.136225,0.111618,0.053526,0.093354


В test есть источники, которых нет в train. Поэтому `source` не использовался как основной признак финальной transformer-модели: он может переобучать модель под train.

In [10]:
experiment_results = pd.DataFrame([
    ["TF-IDF ensemble", "0.05234", "классический ML-бейзлайн"],
    ["Single split RuBERT", "0.04382", "сильный transformer-бейзлайн на одном разбиении"],
    ["5-fold RuBERT", "0.04208", "более устойчивый вариант с кросс-валидацией"],
    ["5-fold RuBERT + rates_more23_soft", "0.04129", "лучший public score среди проведённых экспериментов"],
    ["SVC rerank", "проверено", "не улучшил выбранную public-ветку"],
    ["Known-combo postprocessing", "проверено", "не улучшил выбранную public-ветку"],
    ["TextCNN blend", "проверено", "дополнительная deep learning модель, не выбрана в финальное решение"],
    ["HeadTail RuBERT view", "проверено", "альтернативное представление текста, не выбрано в финальное решение"],
], columns=["подход", "public_score_или_статус", "комментарий"])

experiment_results

,подход,public_score_или_статус,комментарий
0,TF-IDF ensemble,0.05234,классический ML-бейзлайн
1,Single split RuBERT,0.04382,сильный transformer-бейзлайн на одном разбиении
2,5-fold RuBERT,0.04208,более устойчивый вариант с кросс-валидацией
3,5-fold RuBERT + rates_more23_soft,0.04129,лучший public score среди проведённых экспериментов
4,SVC rerank,проверено,не улучшил выбранную public-ветку
5,Known-combo postprocessing,проверено,не улучшил выбранную public-ветку
6,TextCNN blend,проверено,"дополнительная deep learning модель, не выбрана в финальное решение"
7,HeadTail RuBERT view,проверено,"альтернативное представление текста, не выбрано в финальное решение"


## 4. Предобработка текста

Используется мягкая очистка: убираются HTML-теги, CDATA, HTML entities и лишние пробелы. Агрессивная редакторская чистка не использовалась в финальной ветке, потому что она ухудшала качество на public leaderboard.

In [11]:
def clean_text_v2(s):
    s = "" if pd.isna(s) else str(s)
    s = html.unescape(s)
    s = re.sub(r"<!\[CDATA\[|\]\]>", " ", s)
    s = re.sub(r"<!--.*?-->", " ", s)
    s = re.sub(r"<[^>]*>", " ", s)
    s = re.sub(r"&[a-zA-Z]+;", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def cut_text(s, head=7000, tail=2000):
    s = "" if pd.isna(s) else str(s)
    if len(s) <= head + tail:
        return s
    return s[:head] + " " + s[-tail:]

In [12]:
train["title_clean_v2"] = train["title"].apply(clean_text_v2)
test["title_clean_v2"] = test["title"].apply(clean_text_v2)

train["text_clean_v2"] = train["text"].apply(clean_text_v2).apply(cut_text)
test["text_clean_v2"] = test["text"].apply(clean_text_v2).apply(cut_text)

train["bert_text_raw"] = (
    train["title_clean_v2"].fillna("") + "\n" +
    train["title_clean_v2"].fillna("") + "\n" +
    train["text_clean_v2"].fillna("")
)

test["bert_text_raw"] = (
    test["title_clean_v2"].fillna("") + "\n" +
    test["title_clean_v2"].fillna("") + "\n" +
    test["text_clean_v2"].fillna("")
)

print(train["bert_text_raw"].iloc[0][:700])

Рейтинг регионов по уровню закредитованности населения — 2019
Рейтинг регионов по уровню закредитованности населения — 2019
Средний уровень ria.ru 💰 закредитованности ⚡ content> россиян ✨ вырос > за 📍 2019 🏭 ⁉️ год с 44,9 & copy;до 47,1 %. 🗞️ 💰 ❗ >🎥 ⚡ 📻 ' u> Больше 🕒 class="quote">span> >всего > банкам 📅 должны жители p >Калмыкии (86, 2 %), ⏰ 🔴 🔴 « > 🟢 меньше 💵 👇 ✅ всего — >Ингушетии 📍 (9,9%). 📡 🏭 📰 💵 📉 — Смотрите в инфографике Ria.ru, 📡 ✨ 🧾 ✨ " какая долговая нагрузка 'у населения > >вашего & 📰 региона


## 5. Быстрый TF-IDF baseline

Эта часть оставлена для демонстрации базового классического подхода. Финальный submit строится transformer-моделью ниже.

In [13]:
RUN_TFIDF_BASELINE = True

In [14]:
if RUN_TFIDF_BASELINE:
    X = (
        train["title_clean_v2"].fillna("") + " " +
        train["title_clean_v2"].fillna("") + " " +
        train["text_clean_v2"].fillna("")
    ).values

    X_tr, X_val, y_tr, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        shuffle=True
    )

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=150000,
        sublinear_tf=True,
        lowercase=True
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=150000,
        sublinear_tf=True,
        lowercase=True
    )

    X_tr_tfidf = hstack([
        word_vectorizer.fit_transform(X_tr),
        char_vectorizer.fit_transform(X_tr)
    ])

    X_val_tfidf = hstack([
        word_vectorizer.transform(X_val),
        char_vectorizer.transform(X_val)
    ])

    tfidf_model = OneVsRestClassifier(
        LogisticRegression(
            C=4.0,
            max_iter=3000,
            solver="liblinear",
            random_state=SEED
        )
    )

    tfidf_model.fit(X_tr_tfidf, y_tr)
    valid_proba_tfidf = tfidf_model.predict_proba(X_val_tfidf)
    valid_pred_tfidf = (valid_proba_tfidf >= 0.5).astype(int)

    print("TF-IDF validation hamming loss:", hamming_loss(y_val, valid_pred_tfidf))
    print("TF-IDF validation micro F1:", f1_score(y_val, valid_pred_tfidf, average="micro"))
else:
    print("TF-IDF baseline skipped")

TF-IDF validation hamming loss: 0.045615085303801255
TF-IDF validation micro F1: 0.8404522613065326


## 6. Функции для финальной модели

In [15]:
def find_best_thresholds(proba, true):
    thresholds = []

    for j in range(true.shape[1]):
        best_t = 0.5
        best_loss = 10

        for t in np.arange(0.05, 0.96, 0.01):
            pred_j = (proba[:, j] >= t).astype(int)
            loss = (pred_j != true[:, j]).mean()

            if loss < best_loss:
                best_loss = loss
                best_t = t

        thresholds.append(best_t)

    return np.array(thresholds)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    out = []

    for batch in tqdm(loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(**batch).logits

        out.append(torch.sigmoid(logits).detach().cpu().numpy())

    return np.vstack(out)


def format_target(row):
    return "[" + ",".join(map(str, map(int, row))) + "]"


def save_submit(pred, name="sample_submission"):
    sub = sample_submission.copy()
    sub["target"] = [format_target(row) for row in pred]

    path = f"/kaggle/working/{name}.csv"
    sub.to_csv(path, index=False)

    print("saved:", path)
    print("shape:", sub.shape)
    print("id ok:", sub["id"].equals(sample_submission["id"]))

    display(pd.DataFrame({
        "label": [f"target_{i}" for i in range(5)],
        "pred_share": pred.mean(axis=0),
        "train_share": y.mean(axis=0)
    }))

    display(sub["target"].value_counts().head(25))

    return path


def apply_rate_thresholds(proba, target_rates):
    thresholds = []

    for j, rate in enumerate(target_rates):
        t = np.quantile(proba[:, j], 1 - rate)
        thresholds.append(t)

    thresholds = np.array(thresholds)
    pred = (proba >= thresholds).astype(int)

    return pred, thresholds

## 7. Dataset и загрузка RuBERT

Модель: `DeepPavlov/rubert-base-cased`.

Если эта ячейка падает с ошибкой `Temporary failure in name resolution`, в Kaggle нужно проверить `Internet: On` и перезапустить сессию/ячейку. Это не ошибка кода, а проблема доступа Kaggle к Hugging Face.

In [16]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"

MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
N_FOLDS = 5

# Веса не хранятся в репозитории, модель загружается при запуске.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class NewsDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in item.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item


test_ds = NewsDataset(test["bert_text_raw"].values)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0
)

print("dataset ready")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

dataset ready


## 8. Финальная модель: 5-fold RuBERT

На каждом fold модель обучается отдельно. Для test усредняются вероятности по пяти моделям.

In [17]:
gc.collect()
torch.cuda.empty_cache()

texts = train["bert_text_raw"].values
labels = y.astype(np.float32)

oof_proba_5fold = np.zeros((len(train), 5), dtype=np.float32)
test_proba_5fold_sum = np.zeros((len(test), 5), dtype=np.float32)

stratify_col = train["labels_count"].copy()
stratify_col = stratify_col.where(stratify_col < 3, 3)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

fold_scores = []

In [ ]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(texts, stratify_col), 1):
    print()
    print("=" * 60)
    print("fold:", fold)
    print("=" * 60)

    gc.collect()
    torch.cuda.empty_cache()

    train_ds = NewsDataset(texts[tr_idx], labels[tr_idx])
    valid_ds = NewsDataset(texts[val_idx], labels[val_idx])

    g = torch.Generator()
    g.manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=g
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=0
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=5,
        problem_type="multi_label_classification"
    )

    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.01
    )

    num_training_steps = EPOCHS * len(train_loader)
    num_warmup_steps = int(num_training_steps * 0.1)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    criterion = torch.nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_loss = 10
    best_state = None
    best_thresholds = None
    best_valid_proba = None
    best_epoch = 0

    for epoch in range(EPOCHS):
        model.train()
        losses = []

        for batch in tqdm(train_loader, desc=f"fold {fold} epoch {epoch + 1}/{EPOCHS}"):
            labels_batch = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(**batch).logits
                loss = criterion(logits, labels_batch)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(loss.item())

        valid_proba = predict_proba(model, valid_loader)
        thresholds = find_best_thresholds(valid_proba, labels[val_idx])
        valid_pred = (valid_proba >= thresholds).astype(int)
        valid_loss = hamming_loss(labels[val_idx], valid_pred)

        print()
        print("fold:", fold)
        print("epoch:", epoch + 1)
        print("train_loss:", np.mean(losses))
        print("valid hamming loss:", valid_loss)
        print("thresholds:", thresholds)
        print("true rates:", labels[val_idx].mean(axis=0))
        print("pred rates:", valid_pred.mean(axis=0))

        if valid_loss < best_loss:
            best_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_thresholds = thresholds.copy()
            best_valid_proba = valid_proba.copy()
            best_epoch = epoch + 1
            print("new best fold epoch")

    model.load_state_dict(best_state)
    model = model.to(device)

    oof_proba_5fold[val_idx] = best_valid_proba

    test_proba_fold = predict_proba(model, test_loader)
    test_proba_5fold_sum += test_proba_fold / N_FOLDS

    np.save(f"/kaggle/working/rubert_5fold_valid_proba_fold_{fold}.npy", best_valid_proba)
    np.save(f"/kaggle/working/rubert_5fold_test_proba_fold_{fold}.npy", test_proba_fold)

    fold_scores.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "best_loss": best_loss,
        "thresholds": best_thresholds
    })

    print("fold done:", fold)
    print("best epoch:", best_epoch)
    print("best loss:", best_loss)

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

np.save("/kaggle/working/oof_proba_rubert_5fold.npy", oof_proba_5fold)
np.save("/kaggle/working/test_proba_rubert_5fold.npy", test_proba_5fold_sum)

fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

print("5-fold done")
print("oof:", oof_proba_5fold.shape)
print("test:", test_proba_5fold_sum.shape)


fold: 1


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

...


fold: 4
epoch: 1
train_loss: 0.22759243850841335
valid hamming loss: 0.04796407185628743
thresholds: [0.54 0.71 0.63 0.34 0.36]
true rates: [0.43323353 0.1269461  0.11556886 0.0742515  0.03053892]
pred rates: [0.46077844 0.10538922 0.08802395 0.05688623 0.02125749]
new best fold epoch


fold 4 epoch 2/3:   0%|          | 0/836 [00:00<?, ?it/s]

  0%|          | 0/105 [00:00<?, ?it/s]


fold: 4
epoch: 2
train_loss: 0.1244167656677833
valid hamming loss: 0.045688622754491016
thresholds: [0.4  0.62 0.65 0.73 0.68]
true rates: [0.43323353 0.1269461  0.11556886 0.0742515  0.03053892]
pred rates: [0.44580838 0.11556886 0.10239521 0.05299401 0.02215569]
new best fold epoch


fold 4 epoch 3/3:   0%|          | 0/836 [00:00<?, ?it/s]

## 9. OOF thresholds

In [ ]:
thresholds_5fold = find_best_thresholds(oof_proba_5fold, y)
pred_oof_5fold = (oof_proba_5fold >= thresholds_5fold).astype(int)

loss_5fold = hamming_loss(y, pred_oof_5fold)

print("OOF 5-fold hamming loss:", loss_5fold)
print("OOF 5-fold hamming score:", 1 - loss_5fold)
print("thresholds:", thresholds_5fold)

display(pd.DataFrame({
    "label": [f"target_{i}" for i in range(5)],
    "threshold": thresholds_5fold,
    "true_share": y.mean(axis=0),
    "oof_pred_share": pred_oof_5fold.mean(axis=0)
}))

## 10. Базовый submit от 5-fold RuBERT

In [ ]:
test_pred_5fold = (test_proba_5fold_sum >= thresholds_5fold).astype(int)

path_anchor = save_submit(
    test_pred_5fold,
    "sample_submission_rubert_5fold_anchor"
)

## 11. Финальная мягкая калибровка долей классов

По OOF-предсказаниям модель немного недооценивала `target_2` и `target_3`, поэтому в финальной ветке была использована мягкая калибровка долей этих классов на test.

In [ ]:
base_rates = test_pred_5fold.mean(axis=0)
print("base rates:", base_rates)

rates_more23_soft = base_rates + np.array([
    -0.004,
    -0.002,
     0.006,
     0.005,
     0.000
])

rates_more23_soft = np.clip(rates_more23_soft, 0.001, 0.999)

pred_more23_soft, thr_more23_soft = apply_rate_thresholds(
    test_proba_5fold_sum,
    rates_more23_soft
)

print("thresholds:", thr_more23_soft)
print("rates:", pred_more23_soft.mean(axis=0))

path_more23_soft = save_submit(
    pred_more23_soft,
    "rubert_5fold_rates_more23_soft"
)

## 12. Сохранение финального файла `sample_submission.csv`

Именно этот файл должен появиться после `Run All`.

In [ ]:
submission = sample_submission.copy()
submission["target"] = [format_target(row) for row in pred_more23_soft]

submission.to_csv("/kaggle/working/sample_submission.csv", index=False)

print("saved final file: /kaggle/working/sample_submission.csv")
print(submission.shape)
print("id ok:", submission["id"].equals(sample_submission["id"]))

display(submission.head())
display(submission["target"].value_counts().head(30))

## 13. Проверка формата

In [ ]:
check = pd.read_csv("/kaggle/working/sample_submission.csv")

print("shape:", check.shape)
print("id ok:", check["id"].equals(sample_submission["id"]))
print("missing values:")
display(check.isna().sum())

print("first rows:")
display(check.head())

print("target example:", check["target"].iloc[0])

## 14. Итог

Финальный файл создаётся здесь:

```text
/kaggle/working/sample_submission.csv
```

В репозиторий не добавляются веса модели. Ноутбук сам загружает предобученный RuBERT и дообучает его на данных соревнования.